# 02 Web后端开发 · 第03课：Pydantic数据模型

上一课使用`dict[str, Any]`接收JSON请求体。字典虽然灵活，但不能直接说明必须有哪些字段、每个字段是什么类型、哪些字段可以不传。Pydantic用模型把这些数据规则写成代码，FastAPI会据此自动验证请求并生成接口文档。

## 本课目标

1. 理解Pydantic模型解决什么问题；
2. 使用`BaseModel`声明数据字段；
3. 区分必填字段、默认值字段和可空字段；
4. 使用`Field`添加长度和数值限制；
5. 访问模型属性并使用`model_dump()`转换数据；
6. 理解FastAPI如何自动验证请求体并返回422；
7. 区分请求模型和响应模型；
8. 使用`response_model`约束和过滤响应数据。

## 1. 为什么普通字典还不够？

假设接口收到：

```python
payload = {"username": "小明", "age": 20}
```

仅仅看到参数类型`dict[str, Any]`时，FastAPI和阅读代码的人无法直接知道：

- `username`是不是必填；
- `age`必须是整数还是也允许字符串；
- 用户名是否有长度限制；
- 是否允许缺少年龄；
- 错误数据应该怎样提示；
- 自动文档应该展示什么请求结构。

如果全部手动检查，代码会出现很多重复判断。Pydantic模型就是一份可执行的数据契约：它同时告诉程序、开发者和自动文档“数据应该长什么样”。

## 2. 第一个Pydantic模型

Pydantic模型使用Python类和类型标注声明字段。

In [ ]:
# BaseModel类来自第三方库pydantic
# 继承BaseModel后，Pydantic会读取类中的字段类型并进行数据验证
from pydantic import BaseModel


class UserProfile(BaseModel):
    username: str
    age: int


# UserProfile(username=..., age=...)：创建并验证一个模型实例
#   username参数对应模型中的username字段
#   age参数对应模型中的age字段
#   返回值：通过验证的UserProfile对象
user = UserProfile(username="小明", age=20)

print(user)
print(type(user))
print(user.username)
print(user.age)

`UserProfile`是我们定义的模型类，`user`是这个类的实例。模型实例不是普通字典，因此读取字段通常使用属性：

```python
user.username
```

而不是：

```python
user["username"]
```

Pydantic会在创建模型实例时执行验证。只有数据通过验证，后面的代码才能得到正常模型对象。

## 3. 必填、默认值和可空不是一回事

观察下面三个字段：

In [ ]:
class Article(BaseModel):
    title: str
    published: bool = False
    summary: str | None = None


article = Article(title="Pydantic入门")
print(article)

| 声明 | 能否省略 | 能否传`None` | 含义 |
|---|---|---|---|
| `title: str` | 不能 | 不能 | 必填字符串 |
| `published: bool = False` | 可以 | 不能 | 省略时使用`False` |
| `summary: str \| None = None` | 可以 | 可以 | 省略或显式传`None`都可以 |

特别注意：

```python
summary: str | None
```

表示值可以是字符串或`None`，但因为没有默认值，这个字段本身仍然必须出现在输入中。

```python
summary: str | None = None
```

才同时表示“可以省略”和“可以为`None`”。

## 4. 验证失败时发生什么？

在普通Python代码中直接创建模型，验证失败会抛出`ValidationError`。

In [ ]:
# ValidationError类来自pydantic，表示模型数据验证失败
from pydantic import ValidationError


try:
    UserProfile(username="小明", age="不是数字")
except ValidationError as error:
    # error.errors()：取得结构化错误列表
    #   不需要参数
    #   返回值：由错误字典组成的列表
    print(error.errors())

错误信息通常会包含：

- 错误发生在哪个字段；
- 期望的数据类型或规则；
- 实际收到的输入；
- 错误类型。

在FastAPI接口中，FastAPI会捕获请求数据产生的验证错误，并自动转换成HTTP `422`响应，因此路由处理函数通常不需要自己捕获这类错误。

## 5. 使用`Field`添加字段限制

类型只能说明“这是字符串”或“这是数字”，还不能说明长度和数值范围。`Field`用于给单个模型字段增加规则和文档信息。

In [ ]:
# Field函数来自pydantic，不是fastapi.Field
from pydantic import Field


class ProductCreate(BaseModel):
    # Field(...)：声明该字段没有默认值，因此仍然必填
    #   第1个位置参数default：...表示必填字段
    #   min_length参数：字符串最少字符数
    #   max_length参数：字符串最多字符数
    #   description参数：显示在自动接口文档中的字段说明
    name: str = Field(
        ...,
        min_length=2,
        max_length=30,
        description="商品名称",
    )

    # gt参数表示greater than，即必须严格大于指定值
    price: float = Field(..., gt=0, description="商品价格")

    # ge表示greater than or equal，即大于等于
    # le表示less than or equal，即小于等于
    stock: int = Field(default=0, ge=0, le=10000)


product = ProductCreate(name="键盘", price=299.0)
print(product)

本课使用的约束参数：

| 参数 | 含义 |
|---|---|
| `min_length` | 最小长度 |
| `max_length` | 最大长度 |
| `gt` | 严格大于 |
| `ge` | 大于等于 |
| `lt` | 严格小于 |
| `le` | 小于等于 |
| `description` | 字段文档说明 |

`Field`描述的是数据契约，不是接口处理函数中的手动`if`。Pydantic会在模型创建阶段执行这些约束。

## 6. 模型实例怎样转换成字典？

Pydantic v2使用`model_dump()`把模型实例转换成Python字典。

In [ ]:
# product.model_dump()：把当前Pydantic模型实例转换成Python字典
#   本例不传参数
#   返回值：包含模型字段和值的dict
product_data = product.model_dump()

print(product_data)
print(type(product_data))

方向要分清：

```text
ProductCreate(...)   输入数据 → Pydantic模型实例
product.model_dump() Pydantic模型实例 → Python字典
```

`model_dump()`是Pydantic v2的API。旧教程中可能看到`.dict()`，本课程统一使用当前v2写法。

## 7. FastAPI怎样使用Pydantic请求模型？

当路由函数的参数类型是Pydantic模型时，FastAPI会把它当作JSON请求体。

In [ ]:
from typing import Any

from fastapi import FastAPI


app = FastAPI(title="Pydantic示例")


class UserCreate(BaseModel):
    username: str = Field(..., min_length=2, max_length=20)
    age: int = Field(..., ge=1, le=120)
    bio: str | None = None
    password: str = Field(..., min_length=6)


@app.post("/users", status_code=201)
def create_user(user: UserCreate) -> dict[str, Any]:
    # FastAPI进入函数前已经完成：读取JSON、创建UserCreate实例、验证字段
    # 所以这里的user不是字典，而是验证通过的UserCreate对象
    data = user.model_dump()
    data["id"] = 1
    return data

客户端提交：

```json
{
  "username": "小明",
  "age": 20,
  "password": "secret123"
}
```

请求到达后的流程：

```text
JSON请求体
   ↓ FastAPI解析JSON
Python数据
   ↓ Pydantic创建UserCreate并验证
验证成功的user模型实例
   ↓ 才调用create_user(user)
路由函数返回数据
```

如果缺少`username`、年龄超出范围或密码太短，验证会在调用`create_user()`之前失败，FastAPI自动返回`422`。

## 8. 为什么请求模型和响应模型要分开？

客户端创建用户时需要提交密码，但服务器绝不能把密码返回给客户端。因此输入和输出不应该使用完全相同的数据模型。

```text
请求模型UserCreate：客户端允许或必须提交什么
响应模型UserPublic：服务器允许返回什么
```


In [ ]:
class UserPublic(BaseModel):
    id: int
    username: str
    age: int
    bio: str | None = None


# app.post(path, status_code=..., response_model=...)：注册POST路由
#   path参数：接口路径
#   status_code参数：成功响应状态码
#   response_model参数：响应必须符合的Pydantic模型
# FastAPI会使用UserPublic验证、转换并过滤最终响应
@app.post("/safe-users", status_code=201, response_model=UserPublic)
def create_safe_user(user: UserCreate) -> dict[str, Any]:
    data = user.model_dump()
    data["id"] = 1
    # data中虽然还有password，但response_model不会把未声明的password返回给客户端
    return data

`response_model=UserPublic`主要产生三种作用：

1. 验证服务器返回的数据是否符合响应契约；
2. 在Swagger/OpenAPI中展示响应结构；
3. 过滤响应模型没有声明的字段，避免返回多余或敏感数据。

请求验证失败通常说明客户端提交的数据不符合契约，FastAPI返回`422`。响应模型验证失败通常说明服务器代码返回了错误结构，是后端实现问题。

## 9. Swagger中会发生什么变化？

Pydantic模型会被转换成JSON Schema并加入OpenAPI文档。打开`/docs`时，可以看到：

- 请求体必须包含哪些字段；
- 字段类型；
- 哪些字段必填；
- 默认值；
- 长度或数值限制；
- 成功响应的数据结构；
- 验证失败可能返回的422响应。

这就是“数据模型既是验证规则，也是接口文档”的含义。

## 10. 本课整体流程

```text
客户端发送JSON请求体
        ↓
FastAPI解析JSON
        ↓
Pydantic请求模型验证字段
    ├─ 失败 → FastAPI自动返回422，不执行路由函数
    └─ 成功 → 得到模型实例
                    ↓
              执行业务函数
                    ↓
              产生返回数据
                    ↓
Pydantic响应模型验证并过滤
        ↓
FastAPI生成JSON响应
```

## 11. 本课作业：图书创建接口

请完成同目录`app.py`中的`BookCreate`、`BookResponse`和`create_book()`。不要修改类名、函数名、路由路径和测试文件。

### 11.1 客户端请求模型`BookCreate`

| 字段 | 类型 | 是否必填 | 默认值 | 约束 | 来源 |
|---|---|---|---|---|---|
| `title` | 字符串 | 是 | 无 | 长度1至100 | 客户端提交 |
| `price` | 浮点数 | 是 | 无 | 必须严格大于0 | 客户端提交 |
| `in_stock` | 布尔值 | 否 | `True` | 无 | 客户端可提交 |
| `note` | 字符串或`None` | 否 | `None` | 无 | 客户端可提交 |

### 11.2 服务器响应模型`BookResponse`

响应中必须包含：

- 服务器生成的整数`id`，本课固定为`1`；
- 验证后的`title`；
- 验证后的`price`；
- 实际使用的`in_stock`；
- 实际使用的`note`。

`id`不由客户端提交，而是服务器生成。请求体和响应体的数据来源不能混淆。

### 11.3 接口契约

| 方法与路径 | 请求体 | 成功状态码 | 响应模型 |
|---|---|---|---|
| `POST /books` | `BookCreate` | 201 | `BookResponse` |

合法请求示例：

```json
{
  "title": "Python工程实践",
  "price": 59.9
}
```

成功响应中应自动使用`in_stock=true`和`note=null`。缺少必填字段、标题长度不合法、价格不大于0或类型无法解析时，由FastAPI自动返回`422`。

你需要自己决定模型字段如何声明，以及怎样从`BookCreate`实例构造符合`BookResponse`的返回数据。

运行：

```powershell
pytest -v
```

最终目标是`10 passed`。

## 12. 理解题（请把答案写在每题下面）

1. 相比`dict[str, Any]`，Pydantic模型额外提供了哪些能力？

2. 为什么自定义模型类要继承`BaseModel`？创建模型实例时，验证发生在什么时候？

3. `title: str`、`published: bool = False`和`summary: str | None = None`在必填、默认值和可空方面有什么区别？

4. Pydantic模型实例与普通字典有什么区别？`model_dump()`的作用是什么？

5. `Field`中的`min_length`、`max_length`、`gt`和`ge`分别限制什么？

6. FastAPI收到不符合请求模型的数据时，为什么通常不会执行路由处理函数？

7. 为什么创建用户时，请求模型可以包含`password`，响应模型却不应该包含它？

8. `response_model`除了生成文档外，还会对服务器返回的数据做什么？